# 🏥 Diabetes Readmission Prediction — Complete Pipeline

**Dataset:** UCI Diabetes 130-US Hospitals (1999–2008), 101,766 encounters  
**Goal:** Predict 30-day hospital readmission (binary classification)  
**Phases:** Data → EDA → Preprocessing → Training → Evaluation → Explainability → ONNX Export

> **How to run:** Runtime → Run all  
> Expected total time: ~8–12 min on a free Colab T4 GPU

---
**Key results (pre-computed):**
| Metric | Value |
|--------|-------|
| Best model | LightGBM |
| PR-AUC | 0.2383 (95% CI: 0.222–0.255) |
| ROC-AUC | 0.674 |
| Selected threshold | 0.651 (Precision ≥ 0.30) |
| Recall @ threshold | 22.1% |

**Honest caveat:** The literature ceiling for this dataset is ROC-AUC ~0.68–0.70 (administrative records, no vitals). Our results are within that range and validated with patient-level GroupSplit to prevent leakage.

## 0️⃣ Environment Setup

In [ ]:
# Install all dependencies
!pip install -q scikit-learn lightgbm joblib pandas numpy matplotlib \
               skl2onnx onnxruntime shap tabulate

import sys, os
print(f'Python {sys.version}')

import sklearn, lightgbm, shap, skl2onnx
print(f'sklearn={sklearn.__version__}  lightgbm={lightgbm.__version__}  shap={shap.__version__}  skl2onnx={skl2onnx.__version__}')

In [ ]:
# ── Option A: Clone from GitHub (replace with your repo URL) ──────────────
GITHUB_REPO = 'https://github.com/AbdulrazaqSultan/diabetes-readmission-prediction'

import os
REPO_DIR = '/content/diabetes-readmission-prediction'

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git pull
else:
    !git clone {GITHUB_REPO} {REPO_DIR}
    %cd {REPO_DIR}

sys.path.insert(0, REPO_DIR)
print('Working directory:', os.getcwd())

In [ ]:
# ── Download dataset from UCI (auto) ──────────────────────────────────────
import urllib.request, zipfile, pathlib

RAW_DIR = pathlib.Path('data/raw')
RAW_DIR.mkdir(parents=True, exist_ok=True)

CSV_URL = ('https://archive.ics.uci.edu/ml/machine-learning-databases/'
           '00296/dataset_diabetes.zip')
ZIP_PATH = RAW_DIR / 'dataset_diabetes.zip'

if not (RAW_DIR / 'diabetic_data.csv').exists():
    print('Downloading dataset from UCI...')
    urllib.request.urlretrieve(CSV_URL, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(RAW_DIR)
    # UCI zip puts files in a subfolder — move them up
    for f in RAW_DIR.rglob('*.csv'):
        if f.parent != RAW_DIR:
            f.rename(RAW_DIR / f.name)
    print('Download complete.')
else:
    print('Dataset already present.')

# Verify
for fname in ['diabetic_data.csv', 'IDS_mapping.csv']:
    p = RAW_DIR / fname
    size = p.stat().st_size if p.exists() else 0
    status = '✅' if size > 1000 else '❌'
    print(f'{status} {fname}: {size:,} bytes')

## 1️⃣ Data Loading & EDA

In [ ]:
from src.data_loader import load_and_clean, make_target

df = load_and_clean()
y  = make_target(df)

print(f'Shape: {df.shape}')
print(f'Positive rate (<30 day readmit): {y.mean():.2%}')
print(f'Unique patients: {df["patient_nbr"].nunique():,}')
df.head(3)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
import numpy as np

fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# 1. Target distribution
ax0 = fig.add_subplot(gs[0, 0])
vc = df['readmitted'].value_counts()
bars = ax0.bar(vc.index, vc.values, color=['#2ecc71','#e74c3c','#3498db'])
for bar, val in zip(bars, vc.values):
    ax0.text(bar.get_x()+bar.get_width()/2, bar.get_height()+200,
             f'{val:,}', ha='center', fontsize=9)
ax0.set_title('Readmission Distribution'); ax0.set_ylabel('Count')

# 2. Binary target
ax1 = fig.add_subplot(gs[0, 1])
ax1.pie([y.sum(), (y==0).sum()], labels=['<30 days (pos)', 'No/≥30 (neg)'],
        colors=['#e74c3c','#2ecc71'], autopct='%1.1f%%', startangle=90)
ax1.set_title('Binary Target Split')

# 3. Missing values
ax2 = fig.add_subplot(gs[0, 2])
miss = df.isnull().mean().sort_values(ascending=False).head(15)
colors = ['#e74c3c' if v > 0.4 else '#3498db' for v in miss.values]
ax2.barh(miss.index[::-1], miss.values[::-1], color=colors[::-1])
ax2.axvline(0.4, color='red', ls='--', lw=1, label='40% threshold')
ax2.set_xlabel('Missing fraction'); ax2.set_title('Top-15 Missing Columns')
ax2.legend(fontsize=8)

# 4. Visits per patient
ax3 = fig.add_subplot(gs[1, 0])
visits = df.groupby('patient_nbr').size()
ax3.hist(visits.clip(upper=15), bins=15, color='#9b59b6', edgecolor='white')
ax3.set_xlabel('Visits per patient (capped 15)'); ax3.set_ylabel('Count')
ax3.set_title(f'Patient Visit Distribution (max={visits.max()})')

# 5. Age distribution
ax4 = fig.add_subplot(gs[1, 1])
age_order = ['[0-10)','[10-20)','[20-30)','[30-40)','[40-50)',
             '[50-60)','[60-70)','[70-80)','[80-90)','[90-100)']
age_counts = df['age'].value_counts().reindex(age_order, fill_value=0)
ax4.bar(range(len(age_order)), age_counts.values, color='#1abc9c')
ax4.set_xticks(range(len(age_order))); ax4.set_xticklabels(age_order, rotation=45, ha='right', fontsize=8)
ax4.set_ylabel('Count'); ax4.set_title('Age Distribution')

# 6. time_in_hospital by readmission
ax5 = fig.add_subplot(gs[1, 2])
for label, color in [('<30', '#e74c3c'), ('NO', '#2ecc71'), ('>30', '#3498db')]:
    subset = df[df['readmitted']==label]['time_in_hospital']
    ax5.hist(subset, bins=14, alpha=0.6, label=label, color=color, density=True)
ax5.set_xlabel('Days in Hospital'); ax5.set_ylabel('Density')
ax5.set_title('Time in Hospital by Readmission'); ax5.legend()

fig.suptitle('EDA — UCI Diabetes 130-US Hospitals Dataset', fontsize=14, fontweight='bold')
plt.savefig('reports/figures/eda_overview.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: reports/figures/eda_overview.png')

## 2️⃣ Preprocessing Pipeline

In [ ]:
from src.preprocess import prepare_features, build_preprocessor, map_icd9_to_chapter
import numpy as np

X = prepare_features(df)
print(f'Feature matrix shape (raw): {X.shape}')
print(f'dtypes: {X.dtypes.value_counts().to_dict()}')

# Quick ICD-9 unit test
test_cases = [
    ('250.01', 'Diabetes'), ('428',   'Circulatory'),
    ('V45',   'Other'),     ('E888',  'Injury'),
    (None,    'Other'),
]
print('\nICD-9 chapter mapping check:')
all_ok = True
for code, expected in test_cases:
    result = map_icd9_to_chapter(code)
    ok = result == expected
    all_ok = all_ok and ok
    print(f'  {str(code):10s} → {result:15s}  {"✅" if ok else "❌"}')
print('All ICD-9 tests:', '✅ PASS' if all_ok else '❌ FAIL')

# Fit ColumnTransformer
from sklearn.model_selection import GroupShuffleSplit
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, te_idx = next(gss.split(X, y, groups=df['patient_nbr']))
pre = build_preprocessor()
pre.fit(X.iloc[tr_idx])
Z = pre.transform(X.iloc[te_idx])
print(f'\nTransformed shape: {Z.shape}  (n_features = {Z.shape[1]})')
print(f'NaN after transform: {np.isnan(Z).sum()}')

## 3️⃣ Training & Model Comparison

In [ ]:
# ── Option A: Run training from scratch ───────────────────────────────────
# Uncomment to retrain (adds ~5–8 min):
# from src.train import main as train_main
# train_main()

# ── Option B: Load pre-trained artefacts (default) ────────────────────────
import joblib, json
from pathlib import Path
import numpy as np

MODELS_DIR = Path('models')
pipe = joblib.load(MODELS_DIR / 'model.joblib')
feature_names = np.load(MODELS_DIR / 'feature_names.npy', allow_pickle=True)
idx = np.load(MODELS_DIR / 'split_indices.npz')

X_tr, X_te = X.iloc[idx['train']], X.iloc[idx['test']]
y_tr, y_te = y.iloc[idx['train']], y.iloc[idx['test']]

print(f'Loaded model: {type(pipe.named_steps["clf"]).__name__}')
print(f'n_features: {len(feature_names)}')
print(f'Train: {len(X_tr):,} | Test: {len(X_te):,} | Pos rate (test): {y_te.mean():.2%}')

In [ ]:
# Display model comparison table
metrics_md = Path('reports/metrics.md').read_text()
# Print only the table section
lines = metrics_md.split('\n')
in_table = False
table_lines = []
for line in lines:
    if line.startswith('|'):
        table_lines.append(line)
        in_table = True
    elif in_table and line.startswith('**Selected'):
        table_lines.append(line)
        break

from IPython.display import Markdown, display
display(Markdown('\n'.join(table_lines)))

## 4️⃣ Evaluation & Threshold Selection

In [ ]:
# Run evaluation (generates 4 figures + updates metrics.md)
# Skip if you want to just view existing figures
RUN_EVALUATE = False  # set True to regenerate figures

if RUN_EVALUATE:
    from src.evaluate import main as eval_main
    eval_main()

# Display saved figures
from IPython.display import Image, display
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig_dir = Path('reports/figures')
fig_names = ['pr_curve.png', 'roc_curve.png', 'confusion_matrix.png', 'calibration.png']

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, fname in zip(axes, fig_names):
    img = mpimg.imread(fig_dir / fname)
    ax.imshow(img); ax.axis('off'); ax.set_title(fname.replace('.png',''), fontsize=10)
plt.suptitle('Phase 4 — Evaluation Figures', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Print threshold
threshold_data = json.loads((MODELS_DIR / 'threshold.json').read_text())
print(f"\nSelected threshold : {threshold_data['threshold']:.4f}")
print(f"Clinical rule      : {threshold_data['rule']}")
print(f"Recall             : {threshold_data['recall']:.4f}")
print(f"Precision          : {threshold_data['precision']:.4f}")

## 5️⃣ Explainability

In [ ]:
# Full TreeExplainer SHAP for LightGBM (only possible in Colab with shap installed)
import shap
import numpy as np
from src.preprocess import build_preprocessor

RUN_SHAP = True

if RUN_SHAP:
    pre_fitted = pipe.named_steps['pre']
    clf        = pipe.named_steps['clf']

    # Build preprocessed test matrix
    Z_te   = pre_fitted.transform(X_te)
    names  = pre_fitted.get_feature_names_out()

    # Sample 1000 rows for speed
    rng    = np.random.default_rng(42)
    sample = rng.choice(len(Z_te), size=1000, replace=False)
    Z_s    = Z_te[sample]

    # TreeExplainer (exact, fast for LightGBM)
    explainer  = shap.TreeExplainer(clf)
    shap_vals  = explainer.shap_values(Z_s)
    # For binary classification shap returns list [neg, pos] or single array
    if isinstance(shap_vals, list):
        sv = shap_vals[1]  # positive class
    else:
        sv = shap_vals

    print(f'SHAP values shape: {sv.shape}')

    # Beeswarm (tree SHAP)
    shap.summary_plot(sv, Z_s, feature_names=names, max_display=15,
                      show=False)
    plt.title('TreeExplainer SHAP — LightGBM (top 15)', fontsize=12)
    plt.tight_layout()
    plt.savefig('reports/figures/shap_tree_summary.png', dpi=120, bbox_inches='tight')
    plt.show()

    # Bar plot
    shap.summary_plot(sv, Z_s, feature_names=names, plot_type='bar',
                      max_display=10, show=False)
    plt.title('Top-10 Features — mean |SHAP| (TreeExplainer)', fontsize=12)
    plt.tight_layout()
    plt.savefig('reports/figures/shap_tree_bar.png', dpi=120, bbox_inches='tight')
    plt.show()

    print('✅ TreeExplainer SHAP complete')

In [ ]:
# View pre-computed explainability figures
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
shap_figs = ['shap_summary.png', 'shap_bar.png', 'feature_importance.png']
for ax, fname in zip(axes, shap_figs):
    img = mpimg.imread(fig_dir / fname)
    ax.imshow(img); ax.axis('off'); ax.set_title(fname.replace('.png',''), fontsize=10)
plt.suptitle('Phase 5 — Explainability Figures', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Print top-5 features from explainability.json
exp = json.loads((MODELS_DIR / 'explainability.json').read_text())
print('\nTop-5 (exact linear SHAP):')
for i, item in enumerate(exp['top5_linear_shap'], 1):
    print(f"  {i}. {item['feature']:35s} mean|φ|={item['mean_abs_phi']:.4f}  ({item['direction']} risk)")
print('\nTop-5 (LightGBM permutation importance):')
for i, (feat, val) in enumerate(list(exp['top5_lightgbm_permutation'].items())[:5], 1):
    print(f'  {i}. {feat:35s} ΔPR-AUC={val:.5f}')

## 6️⃣ ONNX Export & Parity Test

In [ ]:
from src.export_onnx import main as export_main, parity_test

# Export model.onnx + feature_schema.json
export_main(check=False)

# Show feature_schema.json
schema = json.loads(Path('models/feature_schema.json').read_text())
print(f"\nfeature_schema.json summary:")
print(f"  model_type      : {schema['model_type']}")
print(f"  n_onnx_features : {schema['n_onnx_features']}")
print(f"  threshold       : {schema['threshold']:.4f}")
print(f"  feature groups  : {list(schema['feature_groups'].keys())}")

In [ ]:
# Parity test: max |sklearn − onnx| < 1e-5
ok = parity_test(pipe, X_te, y_te)
assert ok, 'ONNX parity FAILED — check the export step'
print('\n✅ ONNX parity verified')

In [ ]:
# Demo: single-patient inference with ONNX Runtime
import onnxruntime as rt
import numpy as np

sess = rt.InferenceSession('models/model.onnx')
input_names = [inp.name for inp in sess.get_inputs()]

# Take the first test row
row = X_te.iloc[[0]]
feed = {}
for name in input_names:
    col = row[name]
    if col.dtype == object:
        feed[name] = col.values.reshape(-1,1).astype(bytes)
    else:
        feed[name] = col.values.reshape(-1,1).astype(np.float32)

onnx_out = sess.run(None, feed)
onnx_prob = onnx_out[1][0][1] if not isinstance(onnx_out[1], list) else onnx_out[1][0][1]
sk_prob   = pipe.predict_proba(row)[0, 1]
threshold = json.loads(Path('models/threshold.json').read_text())['threshold']

print(f'Patient #0 (true label: {y_te.iloc[0]})')
print(f'  sklearn probability : {sk_prob:.6f}')
print(f'  ONNX probability    : {onnx_prob:.6f}')
print(f'  |difference|        : {abs(sk_prob - onnx_prob):.2e}')
print(f'  threshold           : {threshold:.4f}')
print(f'  prediction          : {"HIGH RISK (<30 days)" if onnx_prob >= threshold else "low risk"}')

## 📊 Final Report

In [ ]:
from IPython.display import Markdown, display
display(Markdown(Path('reports/metrics.md').read_text()))

In [ ]:
# Final artefacts checklist
checks = [
    'data/raw/diabetic_data.csv',
    'models/model.joblib',
    'models/model.onnx',
    'models/threshold.json',
    'models/feature_schema.json',
    'models/explainability.json',
    'reports/metrics.md',
    'reports/figures/pr_curve.png',
    'reports/figures/roc_curve.png',
    'reports/figures/shap_summary.png',
    'reports/figures/shap_tree_summary.png',
]

print('=== Artefacts Checklist ===')
all_ok = True
for rel in checks:
    p = Path(rel)
    ok = p.exists() and p.stat().st_size > 0
    all_ok = all_ok and ok
    print(f'  {"✅" if ok else "❌"} {rel}')
print()
print('ALL ARTEFACTS PRESENT:', '✅ YES' if all_ok else '❌ NO')